In [ ]:
import scipy.signal as signal
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import textwrap
import os
import sys
import numpy as np
import pandas as pd
import importlib
import h5py

In [ ]:
def print_structure(name, obj):
    print(f"{name} is a {type(obj)}")

with h5py.File('wille_simulated_data.h5', 'r') as f:
    # Assuming the dataset is a 2D array: (number_of_spectra, channels)
    data = f['data/block0_values'][:]

# Define combined statistics from the raw captured spectra
# Axis 0 averages across measurements (rows) producing a single spectrum
combined_mean = np.mean(data, axis=0)
combined_median = np.median(data, axis=0)

n_channels = len(combined_mean)
frequencies = np.linspace(1419.5, 1421.5, n_channels)

plt.figure(figsize=(10, 6))

# --- Plot both lines on the same axes ---
# We use a dashed line for Mean so the Median is visible underneath
plt.plot(frequencies, combined_mean, label='np.mean (Less noisy)', 
         color='blue', linestyle='--', linewidth=2)

plt.plot(frequencies, combined_median, label='np.median (Handles interference)', 
         color='red', alpha=0.6, linewidth=1.5)

# --- Formatting based on your lab requirements ---
plt.title('Simulated Combined Spectral Averaging: Mean vs Median')
plt.xlabel('Frequency (MHz)')
plt.ylabel('Power')
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend()

# --- Analysis check from your lab text ---
# 1. Does the bandpass reach zero at the edges? (Check 1419.5 and 1421.5)
# 2. Is the HI line visible, or is it hidden by the SDR's FIR filter shape?
plt.show()

frequencies = np.linspace(1419.5, 1421.5, data.shape[1])

plt.figure(figsize=(12, 6))

# 1. Plot a few RAW spectra (e.g., the first 3 measurements)
# These will show the high noise level of individual captures
plt.plot(frequencies, data[0, :], label='Raw Spectrum 1', alpha=0.3, color='gray')
plt.plot(frequencies, data[1, :], label='Raw Spectrum 2', alpha=0.3, color='lightgray')

# 2. Overlay your COMBINED Mean for comparison
# This demonstrates how the mean "gives a less noisy result"
plt.plot(frequencies, combined_mean, label='Combined Mean (Processed)', 
         color='blue', linewidth=2)

plt.title('Raw vs. Combined Spectral Data')
plt.xlabel('Frequency (MHz)')
plt.ylabel('Power')
plt.legend()
plt.grid(True, linestyle=':', alpha=0.5)

# Setting a y-limit can help if one raw spike is huge
plt.ylim(np.min(combined_mean), np.max(data[0, :]) * 1.1)

plt.show()

plt.figure(figsize=(10, 8))

# 1. Create the Spectrogram
# data: rows are individual measurements (time), columns are channels (frequency)
im = plt.imshow(data, 
                aspect='auto', 
                extent=[frequencies[0], frequencies[-1], data.shape[0], 0],
                cmap='viridis',
                interpolation='none')

# 2. Add a colorbar to show power intensity
cbar = plt.colorbar(im)
cbar.set_label('Power')

# 3. Labeling
plt.title('Waterfall Plot: Raw Spectra Over Time')
plt.xlabel('Frequency (MHz)')
plt.ylabel('Measurement Number (Time)')

plt.show()


In [ ]:
# Load data from an .npz file and plot a waterfall (measurements x channels)
npz_path = 'bighorntest3.npz'  # change this to the NPZ file you want to use
with np.load(npz_path) as npz:
    # Prefer an entry named 'data' if present, otherwise take the first array
    if 'data' in npz.files:
        npz_data = npz['data']
    else:
        keys = [k for k in npz.files]
        npz_data = npz[keys[0]]

# Convert and squeeze singleton dims
npz_data = np.asarray(npz_data)
npz_data = np.squeeze(npz_data)

# Normalize to 2D: (measurements, channels)
if npz_data.ndim == 1:
    npz_data = npz_data[np.newaxis, :]
elif npz_data.ndim == 3:
    # Try to detect which axis is channels. Prefer matching the HDF5 'data' channels if available
    ref_channels = None
    if 'data' in globals() and isinstance(data, np.ndarray):
        ref_channels = data.shape[1] if data.ndim >= 2 else None
    # If any axis matches ref_channels, move it to last and merge the other two axes into measurements
    if ref_channels is not None and ref_channels in npz_data.shape:
        ch_axis = int(np.where(np.array(npz_data.shape) == ref_channels)[0][0])
        # move channels to last axis
        if ch_axis != 2:
            npz_data = np.moveaxis(npz_data, ch_axis, 2)
        npz_data = npz_data.reshape(-1, npz_data.shape[2])
    else:
        # assume last axis is channels (common: blocks x measurements x channels)
        npz_data = npz_data.reshape(-1, npz_data.shape[2])
elif npz_data.ndim != 2:
    raise ValueError(f'NPZ contains array with {npz_data.ndim} dims; expected 1, 2 or 3')

# If an HDF5 `data` variable exists, try to match its orientation
if 'data' in globals() and isinstance(data, np.ndarray):
    if npz_data.shape == data.T.shape and npz_data.shape != data.shape:
        npz_data = npz_data.T

# Heuristic: if rows > cols it's likely transposed (channels x measurements)
rows, cols = npz_data.shape
if rows > cols:
    npz_data = npz_data.T
    rows, cols = npz_data.shape

# Ensure numeric dtype
if not np.issubdtype(npz_data.dtype, np.number):
    try:
        npz_data = npz_data.astype(float)
    except Exception as e:
        raise TypeError(f'Could not convert NPZ data to numeric dtype: {e}')

print(f'Loaded {npz_path} with shape {npz_data.shape}')

frequencies_npz = np.linspace(1419.5, 1421.5, npz_data.shape[1])
plt.figure(figsize=(10, 8))
im = plt.imshow(npz_data,
                aspect='auto',
                extent=[frequencies_npz[0], frequencies_npz[-1], npz_data.shape[0], 0],
                cmap='viridis',
                interpolation='none')
cbar = plt.colorbar(im)
cbar.set_label('Power')
plt.title(f'Waterfall Plot (NPZ): {npz_path}')
plt.xlabel('Frequency (MHz)')
plt.ylabel('Measurement Number (Time)')
plt.show()
midpoint = npz_data.shape[0] // 2
s_on_data = npz_data[:midpoint, :]
s_off_data = npz_data[midpoint:, :]

# --- 2. AVERAGE (To reduce the "grainy" noise) ---
# Your lab manual notes: "the resulting spectrum will look noisy... 
# combining many gives a less noisy result."
s_on_avg = np.mean(s_on_data, axis=0)
s_off_avg = np.mean(s_off_data, axis=0)

# --- 3. REVEAL THE SIGNAL (Difference and Ratio) ---
# Difference: Removes the "fixed pattern" vertical stripes
difference = s_on_avg - s_off_avg

# Ratio: Often cleaner as it accounts for gain variations
# We subtract 1 to center the baseline at 0
ratio = (s_on_avg / s_off_avg) - 1

# --- 4. PLOTTING ---
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10), sharex=True)

# Plot Difference Spectrum
ax1.plot(frequencies_npz, difference, color='tab:red', label='ON - OFF (Difference)')
ax1.set_ylabel('Power Difference')
ax1.set_title('Revealing the HI Line: Difference vs. Ratio')
ax1.grid(True, alpha=0.3)
ax1.legend()

# Plot Ratio Spectrum
ax2.plot(frequencies_npz, ratio, color='tab:blue', label='(ON / OFF) - 1')
ax2.set_xlabel('Frequency (MHz)')
ax2.set_ylabel('Relative Intensity')
ax2.grid(True, alpha=0.3)
ax2.legend()

plt.tight_layout()
plt.show()

from scipy.ndimage import gaussian_filter
baseline = np.median(npz_data, axis=0)

# 2. Subtract the baseline from every row
# This 'flattens' the instrumental hump and removes static vertical noise
diff_waterfall = npz_data - baseline
# 1. Apply a Gaussian blur to the differential data
# sigma=(time_blur, frequency_blur)
# We blur more in time (vertical) to find the consistent signal
smoothed_diff = gaussian_filter(diff_waterfall, sigma=(50, 2))

plt.figure(figsize=(12, 8))

# 2. Re-plot with the smoothed data
im = plt.imshow(smoothed_diff,
                aspect='auto',
                extent=[frequencies_npz[0], frequencies_npz[-1], npz_data.shape[0], 0],
                cmap='RdBu_r', 
                interpolation='bilinear')

# 3. Aggressively tighten the color limits
# This 'squashes' the noise and forces faint signals to show color
v_limit = np.std(smoothed_diff) * 1.5 
im.set_clim(-v_limit, v_limit)

plt.colorbar(im, label='Smoothed Relative Power')
plt.title('Revealing the HI Smear (Gaussian Smoothed)')
plt.show()



# 3. Apply a small amount of Smoothing (optional but helps with the 'smear')
# If the data is too grainy, the line stays hidden. 
# Here we use a basic Gaussian filter if you have scipy, or just show raw.

plt.figure(figsize=(12, 8))

# Use a 'Diverging' colormap (like RdBu_r or coolwarm) 
# This makes features 'brighter' than the average look red/yellow 
# and 'darker' look blue.
norm = colors.TwoSlopeNorm(vcenter=0) # Centers the color scale at 0

im = plt.imshow(diff_waterfall,
                aspect='auto',
                extent=[frequencies_npz[0], frequencies_npz[-1], npz_data.shape[0], 0],
                cmap='RdBu_r', 
                norm=norm,
                interpolation='bilinear') # 'bilinear' helps create that 'smear' look

cbar = plt.colorbar(im)
cbar.set_label('Power relative to Median')
plt.xlim(1420.1, 1420.8)

# Even heavier vertical smoothing to kill the noise
smoothed_diff_heavy = gaussian_filter(diff_waterfall, sigma=(100, 1))

# Re-run your imshow with these tighter limits
im.set_data(smoothed_diff_heavy)

# 2. Re-calculate v_limit based on the SMOOTHED data
# Using 2 or 3 standard deviations of the smoothed data helps features "pop"
v_limit_smoothed = np.std(smoothed_diff_heavy) * 2

# 3. Apply the new limits (This replaces the previous im.set_clim and v_limit lines)
im.set_clim(-v_limit_smoothed, v_limit_smoothed)

# 4. Ensure the x-axis zoom is applied correctly
plt.xlim(1420.1, 1420.8)

plt.show()

In [ ]:
# Compute temporal power spectra (Welch) for NPZ readings
# Uses the existing `npz_data` if present, otherwise loads the NPZ file
npz_path = 'bighorntest3.npz'
try:
    _npz = npz_data
except NameError:
    with np.load(npz_path) as npz:
        if 'data' in npz.files:
            _npz = npz['data']
        else:
            _npz = npz[npz.files[0]]

# Normalize and reshape like the waterfall cell does
_npz = np.asarray(_npz)
_npz = np.squeeze(_npz)
if _npz.ndim == 3:
    _npz = _npz.reshape(-1, _npz.shape[-1])
if _npz.ndim == 1:
    _npz = _npz[np.newaxis, :]
if _npz.ndim != 2:
    raise ValueError(f'Unexpected array shape for PSD computation: {_npz.shape}')

# Choose channel (center) and compute Welch PSD (temporal) for that channel
ch_idx = _npz.shape[1] // 2
timeseries = _npz[:, ch_idx]
from scipy.signal import welch
fs = 1.0  # sampling rate (samples per measurement); set if you know the real rate
nperseg = min(256, max(8, len(timeseries)))
f, Pxx = welch(timeseries, fs=fs, nperseg=nperseg)
plt.figure(figsize=(8, 4))
plt.semilogy(f, Pxx, label=f'Channel {ch_idx}')
plt.title('Welch PSD — single channel')
plt.xlabel('Frequency (cycles per sample)')
plt.ylabel('PSD')
plt.grid(True, which='both', ls=':')
plt.legend()

# Average PSD across a small band around the center channel
halfband = 5
start = max(0, ch_idx - halfband)
stop = min(_npz.shape[1], ch_idx + halfband + 1)
psd_list = []
for c in range(start, stop):
    _, p = welch(_npz[:, c], fs=fs, nperseg=nperseg)
    psd_list.append(p)
psd_mean = np.mean(psd_list, axis=0)
plt.figure(figsize=(8, 4))
plt.semilogy(f, psd_mean, color='tab:orange', label=f'Avg channels {start}-{stop-1}')
plt.title('Welch PSD — averaged band')
plt.xlabel('Frequency (cycles per sample)')
plt.ylabel('PSD')
plt.grid(True, which='both', ls=':')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
on_target_spectrum = np.mean(smoothed_diff_heavy[:midpoint, :], axis=0)

# 2. Create the plot
plt.figure(figsize=(10, 6))

# Plot the 1D Power Spectrum
plt.plot(frequencies_npz, on_target_spectrum, color='red', linewidth=2, label='Smoothed HI Signal')

# Formatting for your report
plt.axhline(0, color='black', linewidth=1, alpha=0.5) # Baseline
plt.axvline(1420.406, color='blue', linestyle='--', alpha=0.7, label='HI Rest Freq')

# Zoom in on the area where you saw the red/blue smear
plt.xlim(1420.2, 1420.7) 

plt.title('Power Spectrum of Isolated HI Signal')
plt.xlabel('Frequency (MHz)')
plt.ylabel('Relative Power (Smoothed)')
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

In [ ]:
from scipy.ndimage import uniform_filter1d

# 1. Define midpoint if not already defined
midpoint = npz_data.shape[0] // 2 

# 2. Get the ON-target spectrum from your differential data
# This is the 1D line that contains your "red" smear
on_target_raw = np.mean(diff_waterfall[:midpoint, :], axis=0)

# 3. Create 'smoothed_peak'
# A size of 10-15 is a good "compromise" for the 1 km/s rule
smoothed_peak = uniform_filter1d(on_target_raw, size=15)

# 4. Constants for Doppler conversion
f0 = 1420.406  # Rest frequency in MHz
c = 299792.458 # km/s

# 5. Convert frequencies to Velocity (km/s)
velocities = c * (f0 - frequencies_npz) / f0

# 6. Plotting
plt.figure(figsize=(10, 6))
plt.plot(velocities, smoothed_peak, color='darkred', linewidth=2, label='HI Profile')

plt.axvline(0, color='black', linestyle='--', alpha=0.5, label='Rest (0 km/s)')
plt.xlim(-150, 150) # Galaxy-standard zoom
plt.axhline(0, color='black', linewidth=0.5)

plt.title('HI Line Velocity Profile')
plt.xlabel('Velocity (km/s)')
plt.ylabel('Relative Power')
plt.grid(True, alpha=0.2)
plt.legend()
plt.show()